# RainNet: Rainfall Intensity Prediction using Deep Neural Networks
PyTorch regression notebook for the Rain Rate Prediction dataset.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
TRAIN_PATH="/kaggle/input/rain-rate-prediction-training-dataset/train.parquet"
VALID_PATH="/kaggle/input/rain-rate-prediction-training-dataset/validation.parquet"
TEST_PATH="/kaggle/input/rain-rate-prediction-training-dataset/test.parquet"

train_df=pd.read_parquet(TRAIN_PATH)
valid_df=pd.read_parquet(VALID_PATH)
test_df=pd.read_parquet(TEST_PATH)

train_df.head()


In [ ]:
TARGET="rain_rate_mm_per_hr"

X_train=train_df.drop(columns=[TARGET]).copy()
y_train=train_df[TARGET].copy()

X_valid=valid_df.drop(columns=[TARGET]).copy()
y_valid=valid_df[TARGET].copy()

X_test=test_df.drop(columns=[TARGET]).copy()
y_test=test_df[TARGET].copy()

# Drop timestamp
for df in (X_train,X_valid,X_test):
    if "timestamp" in df.columns:
        df.drop(columns=["timestamp"], inplace=True)

# Encode object columns automatically
cat_cols=X_train.select_dtypes(include=["object"]).columns
for col in cat_cols:
    mapping={v:i for i,v in enumerate(X_train[col].astype(str).unique())}
    X_train[col]=X_train[col].astype(str).map(mapping)
    X_valid[col]=X_valid[col].astype(str).map(mapping).fillna(-1).astype(int)
    X_test[col]=X_test[col].astype(str).map(mapping).fillna(-1).astype(int)

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_valid=scaler.transform(X_valid)
X_test=scaler.transform(X_test)

X_train=torch.tensor(X_train,dtype=torch.float32)
X_valid=torch.tensor(X_valid,dtype=torch.float32)
X_test=torch.tensor(X_test,dtype=torch.float32)

y_train=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
y_valid=torch.tensor(y_valid.values,dtype=torch.float32).view(-1,1)
y_test=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)


In [ ]:
class RainDataset(Dataset):
    def __init__(self,X,y):
        self.X=X
        self.y=y
    def __len__(self):
        return len(self.X)
    def __getitem__(self,idx):
        return self.X[idx], self.y[idx]

train_loader=DataLoader(RainDataset(X_train,y_train),batch_size=1024,shuffle=True)
valid_loader=DataLoader(RainDataset(X_valid,y_valid),batch_size=1024)
test_loader=DataLoader(RainDataset(X_test,y_test),batch_size=1024)


In [ ]:
class RainNet(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(input_dim,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )
    def forward(self,x):
        return self.net(x)

model=RainNet(X_train.shape[1]).to(device)
criterion=nn.MSELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="min",patience=3)


In [ ]:
train_losses=[]
valid_losses=[]

for epoch in range(20):
    model.train()
    running=0.0
    for X,y in train_loader:
        X,y=X.to(device),y.to(device)
        optimizer.zero_grad()
        pred=model(X)
        loss=criterion(pred,y)
        loss.backward()
        optimizer.step()
        running+=loss.item()

    train_loss=running/len(train_loader)

    model.eval()
    running=0.0
    with torch.no_grad():
        for X,y in valid_loader:
            X,y=X.to(device),y.to(device)
            running+=criterion(model(X),y).item()

    valid_loss=running/len(valid_loader)
    scheduler.step(valid_loss)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f"Epoch {epoch+1:02d} | Train {train_loss:.4f} | Valid {valid_loss:.4f}")


In [ ]:
model.eval()
preds=[]
targets=[]

with torch.no_grad():
    for X,y in test_loader:
        X=X.to(device)
        out=model(X).cpu().numpy()
        preds.extend(out.flatten())
        targets.extend(y.numpy().flatten())

rmse=np.sqrt(mean_squared_error(targets,preds))
mae=mean_absolute_error(targets,preds)
r2=r2_score(targets,preds)

print("RMSE:",rmse)
print("MAE :",mae)
print("R2  :",r2)


In [ ]:
torch.save(model.state_dict(),"RainNet_best_model.pth")

plt.figure(figsize=(8,5))
plt.plot(train_losses,label="Train")
plt.plot(valid_losses,label="Validation")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()
